# Lexical Diversity Analysis

This notebook evaluates lexical diversity across three datasets using:

1. **Type-Token Ratio (TTR)** - Measures vocabulary variation within a dataset
2. **Jaccard Index** - Quantifies lexical overlap between pairs of datasets

**Datasets:**
- **Original (Org)**: Original training dataset
- **Generated (Gen)**: Synthetic dataset generated via LLM
- **Paraphrased (Para)**: Synthetic dataset via paraphrasing


## 1. Setup

In [1]:
%pip install -q pandas transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from typing import List, Set
from transformers import AutoTokenizer
from IPython.display import display

# Load CafeBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("uitnlp/CafeBERT")

C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--uitnlp--CafeBERT. Caching files will still work but in a degraded version that might require more space on your disk. This wa

ImportError: 
 requires the protobuf library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/protocolbuffers/protobuf/tree/master/python#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.


## 2. Lexical Diversity Functions

In [ ]:
def get_tokens(texts: List[str]) -> List[str]:
    """Tokenize texts using CafeBERT tokenizer."""
    tokens = []
    for text in texts:
        if pd.isna(text):
            continue
        words = tokenizer.tokenize(str(text).lower())
        tokens.extend(words)
    return tokens

def get_types(tokens: List[str]) -> Set[str]:
    """Get unique word types from tokens."""
    return set(tokens)

def compute_ttr(texts: List[str]) -> dict:
    """
    Compute Type-Token Ratio (TTR).
    TTR = Number of Unique Words (Types) / Total Number of Words (Tokens)
    
    Returns:
        dict with types, tokens, and TTR value
    """
    tokens = get_tokens(texts)
    types = get_types(tokens)
    
    num_tokens = len(tokens)
    num_types = len(types)
    ttr = num_types / num_tokens if num_tokens > 0 else 0
    
    return {
        'types': num_types,
        'tokens': num_tokens,
        'ttr': ttr
    }

def compute_jaccard(texts_a: List[str], texts_b: List[str]) -> dict:
    """
    Compute Jaccard Index between two datasets.
    J(A, B) = |A ∩ B| / |A ∪ B|
    
    Args:
        texts_a: Original dataset texts
        texts_b: Synthetic dataset texts
        
    Returns:
        dict with intersection, union sizes and Jaccard index
    """
    types_a = get_types(get_tokens(texts_a))
    types_b = get_types(get_tokens(texts_b))
    
    intersection = types_a & types_b
    union = types_a | types_b
    
    jaccard = len(intersection) / len(union) if len(union) > 0 else 0
    
    return {
        'types_a': len(types_a),
        'types_b': len(types_b),
        'intersection': len(intersection),
        'union': len(union),
        'jaccard': jaccard
    }

## 3. Load Data

In [ ]:
# Load three datasets
import os

base_path = 'data/processed'
org_path = os.path.join(base_path, 'train_org_processed.csv')
gen_path = os.path.join(base_path, 'train_1071_gen_processed.csv')
para_path = os.path.join(base_path, 'train_1071_para_processed.csv')

org_df = pd.read_csv(org_path)
gen_df = pd.read_csv(gen_path)
para_df = pd.read_csv(para_path)

print(f"Original dataset: {len(org_df)} samples")
print(f"Generated dataset: {len(gen_df)} samples")
print(f"Paraphrased dataset: {len(para_df)} samples")
print(f"\nColumns in datasets:")
print(f"  Org: {list(org_df.columns)}")
print(f"  Gen: {list(gen_df.columns)}")
print(f"  Para: {list(para_df.columns)}")

Original dataset: 5548 samples
Synthetic dataset: 17491 samples


## 4. Compute Lexical Diversity Metrics

In [ ]:
# Extract texts - try 'Sentence' first, fallback to 'Sentence_clean'
text_column = 'Sentence' if 'Sentence' in org_df.columns else 'Sentence_clean'
org_texts = org_df[text_column].tolist()
gen_texts = gen_df[text_column].tolist()
para_texts = para_df[text_column].tolist()

# Compute TTR for all three datasets
print("Computing TTR for all datasets...")
ttr_org = compute_ttr(org_texts)
ttr_gen = compute_ttr(gen_texts)
ttr_para = compute_ttr(para_texts)

print("\n=== Type-Token Ratio (TTR) ===")
print(f"\nOriginal Dataset:")
print(f"  Types (unique words): {ttr_org['types']:,}")
print(f"  Tokens (total words): {ttr_org['tokens']:,}")
print(f"  TTR: {ttr_org['ttr']:.4f}")

print(f"\nGenerated Dataset:")
print(f"  Types (unique words): {ttr_gen['types']:,}")
print(f"  Tokens (total words): {ttr_gen['tokens']:,}")
print(f"  TTR: {ttr_gen['ttr']:.4f}")

print(f"\nParaphrased Dataset:")
print(f"  Types (unique words): {ttr_para['types']:,}")
print(f"  Tokens (total words): {ttr_para['tokens']:,}")
print(f"  TTR: {ttr_para['ttr']:.4f}")

Computing TTR...

=== Type-Token Ratio (TTR) ===

Original Dataset:
  Types (unique words): 2,651
  Tokens (total words): 91,477
  TTR: 0.0290

Synthetic Dataset:
  Types (unique words): 2,954
  Tokens (total words): 330,581
  TTR: 0.0089


In [ ]:
# Compute Jaccard Index for all pairwise comparisons
print("\nComputing Jaccard Index for all pairs...")

jaccard_gen_para = compute_jaccard(gen_texts, para_texts)
jaccard_gen_org = compute_jaccard(gen_texts, org_texts)
jaccard_para_org = compute_jaccard(para_texts, org_texts)

print("\n=== Jaccard Index Comparisons ===")

print(f"\n1. Generated vs Paraphrased:")
print(f"   Gen types: {jaccard_gen_para['types_a']:,}")
print(f"   Para types: {jaccard_gen_para['types_b']:,}")
print(f"   Intersection: {jaccard_gen_para['intersection']:,}")
print(f"   Union: {jaccard_gen_para['union']:,}")
print(f"   Jaccard Index: {jaccard_gen_para['jaccard']:.4f}")

print(f"\n2. Generated vs Original:")
print(f"   Gen types: {jaccard_gen_org['types_a']:,}")
print(f"   Org types: {jaccard_gen_org['types_b']:,}")
print(f"   Intersection: {jaccard_gen_org['intersection']:,}")
print(f"   Union: {jaccard_gen_org['union']:,}")
print(f"   Jaccard Index: {jaccard_gen_org['jaccard']:.4f}")

print(f"\n3. Paraphrased vs Original:")
print(f"   Para types: {jaccard_para_org['types_a']:,}")
print(f"   Org types: {jaccard_para_org['types_b']:,}")
print(f"   Intersection: {jaccard_para_org['intersection']:,}")
print(f"   Union: {jaccard_para_org['union']:,}")
print(f"   Jaccard Index: {jaccard_para_org['jaccard']:.4f}")


Computing Jaccard Index...

=== Jaccard Index ===

Original types: 2,651
Synthetic types: 2,954
Intersection (shared vocabulary): 2,651
Union (combined vocabulary): 2,954
Jaccard Index: 0.8974


In [ ]:
# Summary tables
print("\n=== Summary: TTR Metrics ===")
ttr_summary = pd.DataFrame({
    'Dataset': ['Original', 'Generated', 'Paraphrased'],
    'Types': [ttr_org['types'], ttr_gen['types'], ttr_para['types']],
    'Tokens': [ttr_org['tokens'], ttr_gen['tokens'], ttr_para['tokens']],
    'TTR': [f"{ttr_org['ttr']:.4f}", f"{ttr_gen['ttr']:.4f}", f"{ttr_para['ttr']:.4f}"]
})
print(ttr_summary.to_string(index=False))

print("\n=== Summary: Jaccard Index Comparisons ===")
jaccard_summary = pd.DataFrame({
    'Comparison': ['Gen vs Para', 'Gen vs Org', 'Para vs Org'],
    'Jaccard Index': [
        f"{jaccard_gen_para['jaccard']:.4f}",
        f"{jaccard_gen_org['jaccard']:.4f}",
        f"{jaccard_para_org['jaccard']:.4f}"
    ],
    'Intersection': [
        jaccard_gen_para['intersection'],
        jaccard_gen_org['intersection'],
        jaccard_para_org['intersection']
    ],
    'Union': [
        jaccard_gen_para['union'],
        jaccard_gen_org['union'],
        jaccard_para_org['union']
    ]
})
print(jaccard_summary.to_string(index=False))

# Display formatted tables
display(ttr_summary)
display(jaccard_summary)


=== Summary ===


,Metric,Original,Synthetic,Comparison
0,Types,2651,2954,-
1,Tokens,91477,330581,-
2,TTR,0.0290,0.0089,-
3,Jaccard Index,-,-,0.8974
